# Variational Quantum Circuit

Raw OPM-MEG → Preprocessing → Epoching → Trial tensors → TT decomposition → TT features → Dimensionality reduction → Angle encoding → VQC → Classification

The objective is to classify each MEG trial into one of four task classes: auditory, somatosensory, motor, or resting-state. The important point is that each trial is treated as an individual sample for classification. The TT decomposition is therefore applied separately to each trial rather than decomposing all 5,853 trials into one giant TT tensor.

200 trials

│

├── Trial 1   → 30 channels × 1401 time points

├── Trial 2   → 30 channels × 1401 time points

├── Trial 3   → 30 channels × 1401 time points

│

└── Trial 200 → 30 channels × 1401 time points

Each trial is an individual observation that needs its own representation and its own task label.
The classifier ultimately needs a dataset that looks like:
one row = one trial

After TT decomposition, we have the 3 TT cores.

For example for rank (1, 15, 10, 1) and tensor (200, 30, 1401):
- G1 = 200 × 15
- G2 = 15 × 30 × 10
- G3 = 10 × 1401

These cores contain the information needed to reconstruct the MEG tensor approximately.
We need to turn them into a feature vector that represents the data.

TT compression reduces redundancy while preserving an approximation of the original tensor. PCA/feature selection afterwards is a separate step whose purpose is to make the representation small enough for the quantum circuit.
PCA / feature selection: Finds a small number of directions/features that are useful for the classification problem.

(PCA) is a statistical method. It simplifies complex data by reducing the number of dimensions. It turns many correlated variables into fewer new variables called principal components. These new components keep most of the important information from the original data.

## Preprocess the data

In [10]:
pip install pennylane scikit-learn tensorly openpyxl matplotlib pandas

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install pennylane

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [25]:
from pathlib import Path

import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt

import tensorly as tl
from tensorly.decomposition import tensor_train
from tensorly.tt_tensor import tt_to_tensor

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.preprocessing import MinMaxScaler

import pennylane as qml

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [3]:
DATA_ROOT = Path("../data/preprocessed")
RESULTS_ROOT = Path("../results/vqc")

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

SUBJECTS = [
    "002",
    "005",
    "006",
    "093"
]

TASKS = [
    "auditory",
    "somatosensory",
    "motor",
    "rest"
]

TASK_LABELS = {
    "auditory": 0,
    "somatosensory": 1,
    "motor": 2,
    "rest": 3
}

def find_run_files(subject, task):

    folder = DATA_ROOT / subject / task

    files = sorted(
        folder.glob("*.npz")
    )

    if len(files) == 0:
        raise FileNotFoundError(
            f"No NPZ files found in {folder}"
        )

    return files

def load_run(filepath):

    data = np.load(
        filepath,
        allow_pickle=True
    )

    epochs = data["epochs"]

    return epochs

for subject in SUBJECTS:

    for task in TASKS:

        files = find_run_files(
            subject,
            task
        )

        print(
            f"\n{subject} | {task}"
        )

        for filepath in files:

            epochs = load_run(filepath)

            print(
                f"  {filepath.name}: "
                f"{epochs.shape}"
            )


002 | auditory
  run01_epochs.npz: (200, 30, 1401)
  run02_epochs.npz: (200, 30, 1401)

002 | somatosensory
  run01_epochs.npz: (201, 30, 1401)
  run02_epochs.npz: (204, 30, 1401)

002 | motor
  run01_epochs.npz: (73, 30, 1401)
  run02_epochs.npz: (70, 30, 1401)
  run03_epochs.npz: (74, 30, 1401)

002 | rest
  run01_epochs.npz: (1, 30, 1401)

005 | auditory
  run01_epochs.npz: (200, 30, 1401)
  run02_epochs.npz: (200, 30, 1401)

005 | somatosensory
  run01_epochs.npz: (202, 30, 1401)
  run02_epochs.npz: (198, 30, 1401)

005 | motor
  run01_epochs.npz: (78, 30, 1401)
  run02_epochs.npz: (92, 30, 1401)
  run03_epochs.npz: (76, 30, 1401)

005 | rest
  run01_epochs.npz: (1, 30, 1401)

006 | auditory
  run01_epochs.npz: (200, 30, 1401)
  run02_epochs.npz: (200, 30, 1401)

006 | somatosensory
  run01_epochs.npz: (203, 30, 1401)
  run02_epochs.npz: (204, 30, 1401)

006 | motor
  run01_epochs.npz: (45, 30, 1401)
  run02_epochs.npz: (66, 30, 1401)
  run03_epochs.npz: (55, 30, 1401)

006 | rest

The current code does:
Events → MNE Epochs
for every task.

That works for Auditory, somatosensory and motor:
stimulus → event → epoch

But rest doesn't have stimulus events.
We have approximately 5 minutes of resting recording with the participant simply fixating on a cross.

So current event detection finds essentially one event, resulting in:
(1, 30, 1401)

That's not a meaningful set of individual rest samples for classification.

Instead, for VQC classification we can divide the continuous rest signal into fixed-length windows.

In [8]:
# ============================================================
# PATHS
# ============================================================

LOADED_ROOT = Path("../data/loaded")

SAVE_ROOT = Path("../data/preprocessed_vqc")


files = sorted(
    LOADED_ROOT.rglob("*.npz")
)

print(
    f"Found {len(files)} files"
)


# ============================================================
# VQC EPOCH SETTINGS
# ============================================================

TMIN = -0.2
TMAX = 0.5

REST_WINDOW = 0.7  # seconds


# ============================================================
# PROCESS FILES
# ============================================================

for file in files:

        ## ---------- LOAD DATA ----------

    subject = file.parent.parent.name
    task = file.parent.name
    run = file.stem

    print(subject, task, run)
    
    data = np.load(file, allow_pickle=True)

    signals = data["signals"]
    aux = data["aux"]
    fs = int(data["fs"])
    channel_names = data["channel_names"].tolist()
    positions = data["positions"]
    orientations = data["orientations"]

    print(file.relative_to(LOADED_ROOT))

    print(
        "Signals:",
        signals.shape
    )

    print(
        "Aux:",
        aux.shape
    )

    print(
        "Sampling frequency:",
        fs
    )


    ## ---------- CREATE MNE ----------

    info = mne.create_info(
        ch_names=channel_names,
        sfreq=fs,
        ch_types=["mag"] * len(channel_names)
    )

    raw = mne.io.RawArray(
        signals,
        info
    )

    print(raw)

    ## ---------- FILTERING ----------
     
    raw_filt = raw.copy()
     
    raw_filt.filter(
        l_freq=1.0,
        h_freq=40.0
    )
    
    raw_filt.notch_filter(
        freqs=[60, 120]
    )

    ## ---------- EVENT DETECTION ----------

    task = file.parent.name.lower()
    print(task)
    print(aux.shape)

    if task != "rest":

        if task == "motor":
                trigger = aux[2]
                threshold = 0.5

        else:
            trigger = aux[0]
            threshold = 2.0      
        
        binary = trigger > threshold
    
        onsets = np.where(
            np.diff(binary.astype(int)) == 1
        )[0]
    
        print(
            "Number of events:",
            len(onsets)
        )

        events = np.column_stack(
            [
                onsets,
                np.zeros(
                    len(onsets),
                    dtype=int
                ),
                np.ones(
                    len(onsets),
                    dtype=int
                )
            ]
        )

        print(
            "Events shape:",
            events.shape
        )

        ## ---------- EPOCHING ----------
        
        epochs = mne.Epochs(
            raw_filt,
            events,
            event_id=1,
            tmin=-0.2,
            tmax=0.5,
            baseline=(-0.2, 0),
            preload=True
        )
    
        print(epochs)

        X = epochs.get_data()
        times = epochs.times
        print(
            "Epochs:",
            X.shape
        )


    else:

        # ====================================================
        # RESTING-STATE EPOCHING
        # ====================================================

        print(
            "Creating fixed-length "
            "resting-state windows..."
        )

        # Number of time points per epoch
        N_TIMES = 1401

        # Get filtered continuous data
        rest_data = raw_filt.get_data()

        # Shape: (n_channels, n_samples)

        n_channels, n_samples = rest_data.shape

        # Calculate number of complete windows

        n_windows = (
            n_samples - N_TIMES
        ) // N_TIMES + 1


        print(
            "Number of rest windows:",
            n_windows
        )

        # Extract non-overlapping windows

        X = np.stack(
            [
                rest_data[
                    :,
                    start:start + N_TIMES
                ]
                for start in range(
                    0,
                    n_windows * N_TIMES,
                    N_TIMES
                )
            ]
        )

        # X shape: (n_windows, n_channels, 1401)

        print(
            "Rest windows:",
            X.shape
        )

        times = np.arange(
            N_TIMES
        ) / fs

        # X shape:
        # (n_windows, n_channels, 1401)

    # ========================================================
    # CHECK FINAL SHAPE
    # ========================================================

    print(
        "Final tensor shape:",
        X.shape
    )


    # ========================================================
    # SAVE
    # ========================================================

    SAVE_PATH = (
            SAVE_ROOT /
            subject /
            task /
            f"{run}_epochs.npz"
        )
        
    SAVE_PATH.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    np.savez_compressed(
        SAVE_PATH,
        epochs=X,
        times=epochs.times,
        fs=fs,
        positions=positions,
        orientations=orientations,
        channel_names=np.array(
            channel_names,
            dtype=object
        )
    )

    print(f"Saved: {SAVE_PATH}")
    print(X.shape)

Found 32 files
002 auditory run01
002/auditory/run01.npz
Signals: (30, 856000)
Aux: (1, 856000)
Sampling frequency: 2000
Creating RawArray with float64 data, n_channels=30, n_times=856000
    Range : 0 ... 855999 =      0.000 ...   428.000 secs
Ready.
<RawArray | 30 x 856000 (428.0 s), ~195.9 MiB, data loaded>
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 6601 samples (3.300 s)

Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing 

Task-related epochs were extracted relative to detected experimental events using a −200 ms to +500 ms window. 

Since resting-state recordings contain no repeated stimulus events, the continuous resting-state data were instead segmented into non-overlapping 700 ms windows containing 1401 samples, producing fixed-size samples compatible with the task epochs.

In [14]:
DATA_ROOT = Path("../data/preprocessed_vqc")
RESULTS_ROOT = Path("../results/vqc")

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

SUBJECTS = [
    "002",
    "005",
    "006",
    "093"
]

TASKS = [
    "auditory",
    "somatosensory",
    "motor",
    "rest"
]

TASK_LABELS = {
    "auditory": 0,
    "somatosensory": 1,
    "motor": 2,
    "rest": 3
}

def find_run_files(subject, task):

    folder = DATA_ROOT / subject / task

    files = sorted(
        folder.glob("*.npz")
    )

    if len(files) == 0:
        raise FileNotFoundError(
            f"No NPZ files found in {folder}"
        )

    return files

def load_run(filepath):

    data = np.load(
        filepath,
        allow_pickle=True
    )

    epochs = data["epochs"]

    return epochs

for subject in SUBJECTS:

    for task in TASKS:

        files = find_run_files(
            subject,
            task
        )

        print(
            f"\n{subject} | {task}"
        )

        for filepath in files:

            epochs = load_run(filepath)

            print(
                f"  {filepath.name}: "
                f"{epochs.shape}"
            )


002 | auditory
  run01_epochs.npz: (200, 30, 1401)
  run02_epochs.npz: (200, 30, 1401)

002 | somatosensory
  run01_epochs.npz: (201, 30, 1401)
  run02_epochs.npz: (204, 30, 1401)

002 | motor
  run01_epochs.npz: (73, 30, 1401)
  run02_epochs.npz: (70, 30, 1401)
  run03_epochs.npz: (74, 30, 1401)

002 | rest
  run01_epochs.npz: (462, 30, 1401)

005 | auditory
  run01_epochs.npz: (200, 30, 1401)
  run02_epochs.npz: (200, 30, 1401)

005 | somatosensory
  run01_epochs.npz: (202, 30, 1401)
  run02_epochs.npz: (198, 30, 1401)

005 | motor
  run01_epochs.npz: (78, 30, 1401)
  run02_epochs.npz: (92, 30, 1401)
  run03_epochs.npz: (76, 30, 1401)

005 | rest
  run01_epochs.npz: (438, 30, 1401)

006 | auditory
  run01_epochs.npz: (200, 30, 1401)
  run02_epochs.npz: (200, 30, 1401)

006 | somatosensory
  run01_epochs.npz: (203, 30, 1401)
  run02_epochs.npz: (204, 30, 1401)

006 | motor
  run01_epochs.npz: (45, 30, 1401)
  run02_epochs.npz: (66, 30, 1401)
  run03_epochs.npz: (55, 30, 1401)

006 | 

## TT decomposition of each individual trial

For each individual epoch/trial, we will do:
X_i ∈ R^ 30×1401

and reshape it to:
X_i ∈ R^30×3×467

because:
3×467=1401.

This reshaping is done because Tensor Train decomposition works naturally with a multi-dimensional tensor. It gives us a three-dimensional tensor with dimensions corresponding to the channel dimension and two factors of the temporal dimension.

Then TT decomposition with:
r1 = 15, r2 = 10.

The TT cores are:
G1 ∈ R 30×15
G2 ∈ R 15×3×10
G3 ∈ R 10×467

The resulting representation contains:
30(15)+15(3)(10)+10(467)
= 450 + 450 + 4670 = 5570 parameters.

### TT decomposition & TT feature extraction

In [ ]:
# ============================================================
# PATHS
# ============================================================

DATA_ROOT = Path("../data/preprocessed_vqc")

RESULTS_ROOT = Path("../results/vqc")

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# DATASET
# ============================================================

SUBJECTS = [
    "002",
    "005",
    "006",
    "093"
]

TASKS = [
    "auditory",
    "somatosensory",
    "motor",
    "rest"
]

TASK_LABELS = {
    "auditory": 0,
    "somatosensory": 1,
    "motor": 2,
    "rest": 3
}

TT_RANKS = [
    1,
    15,
    10,
    1
]

# ============================================================
# TT FEATURE FUNCTION
# ============================================================

def extract_tt_features(
    trial,
    ranks
):
    """
    Convert one MEG trial into TT parameters.

    Input:
        trial: (30, 1401)

    Reshape:
        (30, 3, 467)

    TT ranks:
        (1, 15, 10, 1)

    Output:
        flattened TT representation
        shape = (5570,)
    """

    # --------------------------------------------------------
    # Check original shape
    # --------------------------------------------------------

    if trial.shape != (30, 1401):

        raise ValueError(
            f"Unexpected trial shape: "
            f"{trial.shape}"
        )


    # --------------------------------------------------------
    # Reshape temporal dimension
    # --------------------------------------------------------

    tensor = trial.reshape(
        30,
        3,
        467
    )


    # --------------------------------------------------------
    # Convert to TensorLy format
    # --------------------------------------------------------

    tensor = tl.tensor(
        tensor,
        dtype=tl.float64
    )

    # --------------------------------------------------------
    # TT decomposition
    # --------------------------------------------------------

    tt_cores = tensor_train(
        tensor,
        rank=ranks
    )

    # --------------------------------------------------------
    # Flatten all TT cores
    # --------------------------------------------------------

    # Converting the TT cores into features
    # The number of features is: 30(15)+15(3)(10)+10(467) = 5570 TT features
    # They are parameters of the TT representation that collectively describe the original trial.

    features = np.concatenate(
        [
            tl.to_numpy(core).ravel()
            for core in tt_cores
        ]
    )

    return features

# ============================================================
# PROCESS ALL DATA
# ============================================================

X_tt = []
y = []
subjects = []
tasks = []
runs = []
trial_numbers = []

for subject in SUBJECTS:

    for task in TASKS:

        task_folder = (
            DATA_ROOT /
            subject /
            task
        )

        files = sorted(
            task_folder.glob(
                "*_epochs.npz"
            )
        )

        if not files:

            print(
                f"WARNING: no files found: "
                f"{task_folder}"
            )

            continue

        print(
            f"\n{subject} | "
            f"{task} | "
            f"{len(files)} runs"
        )

        label = TASK_LABELS[task]

        for file in files:

            print(
                f"  Processing "
                f"{file.name}"
            )

            data = np.load(
                file,
                allow_pickle=True
            )

            epochs = data["epochs"]

            print(
                f"    Epochs: "
                f"{epochs.shape}"
            )

            # ------------------------------------------------
            # Process every trial separately
            # ------------------------------------------------

            for trial_idx, trial in enumerate(
                epochs
            ):

                features = extract_tt_features(
                    trial,
                    TT_RANKS
                )

                X_tt.append(
                    features
                )

                y.append(
                    label
                )

                subjects.append(
                    subject
                )

                tasks.append(
                    task
                )

                runs.append(
                    file.stem
                )

                trial_numbers.append(
                    trial_idx
                )


# ============================================================
# CONVERT TO NUMPY
# ============================================================

X_tt = np.asarray(
    X_tt,
    dtype=np.float64
)

y = np.asarray(
    y,
    dtype=np.int64
)

subjects = np.asarray(
    subjects
)

tasks = np.asarray(
    tasks
)

runs = np.asarray(
    runs
)

trial_numbers = np.asarray(
    trial_numbers
)

# ============================================================
# CHECK
# ============================================================

print("\n")
print("=" * 70)
print("TT FEATURE DATASET")
print("=" * 70)

print(
    "X_tt shape:",
    X_tt.shape
)

print(
    "y shape:",
    y.shape
)

print(
    "Unique subjects:",
    np.unique(subjects)
)

print(
    "Unique tasks:",
    np.unique(tasks)
)

print(
    "TT ranks:",
    TT_RANKS
)

# ============================================================
# EXPECTED FEATURE COUNT
# ============================================================

expected_features = (
    30 * 15
    + 15 * 3 * 10
    + 10 * 467
)

print(
    "Expected TT parameters:",
    expected_features
)

if X_tt.shape[1] != expected_features:

    raise ValueError(
        "Unexpected TT feature dimension!"
    )

# ============================================================
# SAVE
# ============================================================

save_path = (
    RESULTS_ROOT /
    "tt_features_r15_r10.npz"
)

np.savez_compressed(
    save_path,
    X_tt=X_tt,
    y=y,
    subjects=subjects,
    tasks=tasks,
    runs=runs,
    trial_numbers=trial_numbers,
    tt_ranks=np.asarray(
        [15, 10]
    )
)

print(
    "\nSaved:"
)

print(
    save_path
)


002 | auditory | 2 runs
  Processing run01_epochs.npz
    Epochs: (200, 30, 1401)
  Processing run02_epochs.npz
    Epochs: (200, 30, 1401)

002 | somatosensory | 2 runs
  Processing run01_epochs.npz
    Epochs: (201, 30, 1401)
  Processing run02_epochs.npz
    Epochs: (204, 30, 1401)

002 | motor | 3 runs
  Processing run01_epochs.npz
    Epochs: (73, 30, 1401)
  Processing run02_epochs.npz
    Epochs: (70, 30, 1401)
  Processing run03_epochs.npz
    Epochs: (74, 30, 1401)

002 | rest | 1 runs
  Processing run01_epochs.npz
    Epochs: (462, 30, 1401)

005 | auditory | 2 runs
  Processing run01_epochs.npz
    Epochs: (200, 30, 1401)
  Processing run02_epochs.npz
    Epochs: (200, 30, 1401)

005 | somatosensory | 2 runs
  Processing run01_epochs.npz
    Epochs: (202, 30, 1401)
  Processing run02_epochs.npz
    Epochs: (198, 30, 1401)

005 | motor | 3 runs
  Processing run01_epochs.npz
    Epochs: (78, 30, 1401)
  Processing run02_epochs.npz
    Epochs: (92, 30, 1401)
  Processing run03

Instead of storing all: 
30 × 3 × 467 = 42,030

For one trial:
X ∈ R ^ 30×3×46

TT decomposes this into three cores: G1, G2, G3
With ranks: (1,15,10,1)

The cores are
- G1 ∈ R ^ 1x30×15
- G2 ∈ R ^ 15×3×10
- G3 ∈ R ^ 10×467×1

We represent the trial using: 
1 * 30 * 15 + 15 * 3 * 10 + 10 * 467 * 1 = 450 + 450 + 4670 = 5570 TT parameters

### TT reconstruction test

Calculates:
- Reconstruction Error
- Correlation : how strongly the values in the original and reconstructed trials vary together.

In [19]:
# ============================================================
# PATHS
# ============================================================

DATA_ROOT = Path(
    "../data/preprocessed_vqc"
)

TT_PATH = Path(
    "../results/vqc/tt_features_r15_r10.npz"
)

# ============================================================
# SETTINGS
# ============================================================

SUBJECT = "002"
TASK = "auditory"
RUN = "run01_epochs.npz"

N_TEST_TRIALS = 10

TT_RANKS = [
    1,
    15,
    10,
    1
]

# ============================================================
# LOAD ORIGINAL DATA
# ============================================================

original_path = (
    DATA_ROOT /
    SUBJECT /
    TASK /
    RUN
)

data = np.load(
    original_path,
    allow_pickle=True
)

epochs = data["epochs"]

print("=" * 70)
print("TT RECONSTRUCTION TEST")
print("=" * 70)

print(
    "Original dataset:",
    epochs.shape
)

print(
    "Testing:",
    SUBJECT,
    TASK,
    RUN
)

# ============================================================
# SELECT TRIALS
# ============================================================

n_trials = min(
    N_TEST_TRIALS,
    len(epochs)
)

rng = np.random.default_rng(
    42
)

trial_indices = rng.choice(
    len(epochs),
    size=n_trials,
    replace=False
)

print(
    "Trial indices:",
    trial_indices
)

# ============================================================
# TEST EACH TRIAL
# ============================================================

errors = []
correlations = []

for trial_idx in trial_indices:

    # --------------------------------------------------------
    # Original trial
    # --------------------------------------------------------

    original = epochs[
        trial_idx
    ]

    print(
        f"\nTrial {trial_idx}"
    )

    print(
        "Original shape:",
        original.shape
    )

    # --------------------------------------------------------
    # Reshape
    # --------------------------------------------------------

    tensor = original.reshape(
        30,
        3,
        467
    )

    # --------------------------------------------------------
    # TT decomposition
    # --------------------------------------------------------

    tensor_tl = tl.tensor(
        tensor,
        dtype=tl.float64
    )

    cores = tensor_train(
        tensor_tl,
        rank=TT_RANKS
    )

    # --------------------------------------------------------
    # Reconstruct
    # --------------------------------------------------------

    reconstructed = tt_to_tensor(
        cores
    )

    reconstructed = tl.to_numpy(
        reconstructed
    )

    # --------------------------------------------------------
    # Reshape back
    # --------------------------------------------------------

    reconstructed = (
        reconstructed
        .reshape(30, 1401)
    )

    # --------------------------------------------------------
    # Reconstruction error
    # --------------------------------------------------------

    numerator = np.linalg.norm(
        original - reconstructed
    )

    denominator = np.linalg.norm(
        original
    )

    relative_error = (
        numerator / denominator
    )

    # --------------------------------------------------------
    # Correlation
    # --------------------------------------------------------

    correlation = np.corrcoef(
        original.ravel(),
        reconstructed.ravel()
    )[0, 1]

    errors.append(
        relative_error
    )

    correlations.append(
        correlation
    )

    # --------------------------------------------------------
    # Print
    # --------------------------------------------------------

    print(
        "Reconstructed shape:",
        reconstructed.shape
    )

    print(
        "Relative reconstruction error:",
        f"{relative_error:.6f}"
    )

    print(
        "Correlation:",
        f"{correlation:.6f}"
    )


# ============================================================
# SUMMARY
# ============================================================

errors = np.asarray(
    errors
)

correlations = np.asarray(
    correlations
)

print("\n")
print("=" * 70)
print("SUMMARY")
print("=" * 70)

print(
    "Mean reconstruction error:",
    f"{errors.mean():.6f}"
)

print(
    "Std reconstruction error:",
    f"{errors.std():.6f}"
)

print(
    "Mean correlation:",
    f"{correlations.mean():.6f}"
)

print(
    "Minimum correlation:",
    f"{correlations.min():.6f}"
)

print(
    "Maximum correlation:",
    f"{correlations.max():.6f}"
)

# ============================================================
# PASS / FAIL
# ============================================================

if np.all(np.isfinite(errors)):

    print(
        "\nPASS: reconstruction values "
        "are finite."
    )

else:

    print(
        "\nFAIL: NaN or infinite "
        "reconstruction error."
    )

if np.all(np.isfinite(correlations)):

    print(
        "PASS: correlations are finite."
    )

else:

    print(
        "FAIL: NaN or infinite "
        "correlations."
    )

TT RECONSTRUCTION TEST
Original dataset: (200, 30, 1401)
Testing: 002 auditory run01_epochs.npz
Trial indices: [ 16 148  17 126  85  84 138  18  40 168]

Trial 16
Original shape: (30, 1401)
Reconstructed shape: (30, 1401)
Relative reconstruction error: 0.153426
Correlation: 0.988100

Trial 148
Original shape: (30, 1401)
Reconstructed shape: (30, 1401)
Relative reconstruction error: 0.103286
Correlation: 0.994329

Trial 17
Original shape: (30, 1401)
Reconstructed shape: (30, 1401)
Relative reconstruction error: 0.107583
Correlation: 0.994181

Trial 126
Original shape: (30, 1401)
Reconstructed shape: (30, 1401)
Relative reconstruction error: 0.086209
Correlation: 0.995653

Trial 85
Original shape: (30, 1401)
Reconstructed shape: (30, 1401)
Relative reconstruction error: 0.065222
Correlation: 0.997635

Trial 84
Original shape: (30, 1401)
Reconstructed shape: (30, 1401)
Relative reconstruction error: 0.105911
Correlation: 0.994290

Trial 138
Original shape: (30, 1401)
Reconstructed shape: 

TT decomposition with ranks (15,10) provides a substantially compressed representation while preserving the overall structure of the MEG trials, with a mean relative reconstruction error of approximately 10.5% and mean correlation of approximately 0.994 in the tested trials.

### TT feature validation

In [20]:
# ============================================================
# LOAD
# ============================================================

PATH = Path(
    "../results/vqc/tt_features_r15_r10.npz"
)

data = np.load(
    PATH,
    allow_pickle=True
)

X = data["X_tt"]
y = data["y"]
subjects = data["subjects"]
tasks = data["tasks"]
runs = data["runs"]
trial_numbers = data["trial_numbers"]

# ============================================================
# BASIC TESTS
# ============================================================

print("=" * 70)
print("TT FEATURE VALIDATION")
print("=" * 70)

print(
    "X shape:",
    X.shape
)

print(
    "y shape:",
    y.shape
)

print(
    "Number of subjects:",
    len(np.unique(subjects))
)

print(
    "Subjects:",
    np.unique(subjects)
)

print(
    "Tasks:",
    np.unique(tasks)
)

# ============================================================
# SHAPE
# ============================================================

assert X.ndim == 2

assert X.shape[1] == 5570

assert len(X) == len(y)

assert len(X) == len(subjects)

assert len(X) == len(tasks)

assert len(X) == len(runs)

assert len(X) == len(trial_numbers)


print(
    "\nPASS: dimensions are correct."
)

# ============================================================
# NaN / INF
# ============================================================

print(
    "\nNaN values:",
    np.isnan(X).sum()
)

print(
    "Infinite values:",
    np.isinf(X).sum()
)

assert not np.isnan(X).any()

assert not np.isinf(X).any()

print(
    "PASS: no NaN or infinite values."
)

# ============================================================
# FEATURE STATISTICS
# ============================================================

print("\n")
print("=" * 70)
print("FEATURE STATISTICS")
print("=" * 70)

print(
    "Minimum:",
    X.min()
)

print(
    "Maximum:",
    X.max()
)

print(
    "Mean:",
    X.mean()
)

print(
    "Std:",
    X.std()
)

# ============================================================
# LABEL COUNTS
# ============================================================

print("\n")
print("=" * 70)
print("TASK COUNTS")
print("=" * 70)

for task in np.unique(tasks):

    count = np.sum(
        tasks == task
    )

    print(
        f"{task:15s}: {count}"
    )


# ============================================================
# SUBJECT COUNTS
# ============================================================

print("\n")
print("=" * 70)
print("SUBJECT COUNTS")
print("=" * 70)

for subject in np.unique(subjects):

    count = np.sum(
        subjects == subject
    )

    print(
        f"{subject}: {count}"
    )


print("\n")
print("=" * 70)
print("ALL TESTS PASSED")
print("=" * 70)

TT FEATURE VALIDATION
X shape: (5853, 5570)
y shape: (5853,)
Number of subjects: 4
Subjects: ['002' '005' '006' '093']
Tasks: ['auditory' 'motor' 'rest' 'somatosensory']

PASS: dimensions are correct.

NaN values: 0
Infinite values: 0
PASS: no NaN or infinite values.


FEATURE STATISTICS
Minimum: -0.6932675315876409
Maximum: 0.9963358569799472
Mean: 0.0023520568911343915
Std: 0.06695370168626955


TASK COUNTS
auditory       : 1600
motor          : 850
rest           : 1774
somatosensory  : 1629


SUBJECT COUNTS
002: 1484
005: 1484
006: 1411
093: 1474


ALL TESTS PASSED


## Standardisation & Dimensionality Reduction

The dataset is now:
X_TT ∈ R^ 5853×5570

where:
- 5,853 = total trials
- 5,570 = TT features per trial.

Dimensionality Reduction Options:
- PCA: finds directions of maximum variance, new features that are combinations of the original parameters. PCA is unsupervised. It doesn't use the task labels to decide what information to keep.
- Feature Selection: select the features that have the strongest relationship with the task labels
    - ANOVA F-score
    - mutual information
    - recursive feature elimination
    - L1 regularisation

Principal Component Analysis (PCA):

Suppose one trial is:

x = [f1, f2, ..., f5570]

PCA creates a new representation:

z = [z1, z2, ..., zk]

where k≪5570.

For example:

5570→8.

The zi's are not individual TT features. Instead, each principal component is a weighted combination of the original features:

z1 = w1	​f1 + w2 f2 + ⋯ + w5570 f5570.

1. First, PCA is unsupervised. It doesn't use the task labels to decide what information to keep. It therefore provides a relatively clean dimensionality-reduction step:
2. Second, your TT features are already highly compressed and correlated. You have 5,570 TT parameters, and many of these parameters can be correlated. PCA is specifically designed to transform correlated variables into a smaller set of orthogonal components.
3. Third, PCA gives you a very straightforward way of controlling the number of qubits.

Apply PCA only on Training Set

Leave-One-Subject-Out cross-validation (LOSO)
Instead of having one fixed test subject, we rotate which subject is the test subject.

- TRAIN:
    - 002
    - 005
    - 006
- TEST:
    - 093


- Fold 1 → test 002
- Fold 2 → test 005
- Fold 3 → test 006
- Fold 4 → test 093

In [4]:
# ============================================================
# PATHS
# ============================================================

DATA_PATH = (
    Path("../results/vqc")
    / "tt_features_r15_r10.npz"
)

RESULTS_ROOT = Path(
    "../results/vqc/pca"
)

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# LOAD TT DATA
# ============================================================

data = np.load(
    DATA_PATH,
    allow_pickle=True
)

X_tt = data["X_tt"]
y = data["y"]
subjects = data["subjects"]
tasks = data["tasks"]
runs = data["runs"]
trial_numbers = data["trial_numbers"]

print("=" * 70)
print("TT → STANDARDISATION → PCA")
print("=" * 70)

print(
    "TT features:",
    X_tt.shape
)

print(
    "Labels:",
    y.shape
)

print(
    "Subjects:",
    np.unique(subjects)
)

print(
    "Tasks:",
    np.unique(tasks)
)

# ============================================================
# PCA DIMENSIONS TO TEST
# ============================================================

N_COMPONENTS = [
    4,
    8,
    12,
    16
]

# ============================================================
# SUBJECTS
# ============================================================

SUBJECT_LIST = np.unique(
    subjects
)

# ============================================================
# LOOP OVER LOSO FOLDS
# ============================================================

for test_subject in SUBJECT_LIST:

    print("\n")
    print("=" * 70)

    print(
        "TEST SUBJECT:",
        test_subject
    )

    print("=" * 70)

    # ========================================================
    # TRAIN / TEST MASKS
    # ========================================================

    train_mask = (
        subjects != test_subject
    )

    test_mask = (
        subjects == test_subject
    )

    # ========================================================
    # SPLIT TT FEATURES
    # ========================================================

    X_train = X_tt[
        train_mask
    ]

    X_test = X_tt[
        test_mask
    ]

    # ========================================================
    # SPLIT LABELS
    # ========================================================

    y_train = y[
        train_mask
    ]

    y_test = y[
        test_mask
    ]

    # ========================================================
    # SPLIT METADATA
    # ========================================================

    subjects_train = subjects[
        train_mask
    ]
    subjects_test = subjects[
        test_mask
    ]

    tasks_train = tasks[
        train_mask
    ]
    tasks_test = tasks[
        test_mask
    ]

    runs_train = runs[
        train_mask
    ]
    runs_test = runs[
        test_mask
    ]

    trial_numbers_train = trial_numbers[
        train_mask
    ]
    trial_numbers_test = trial_numbers[
        test_mask
    ]


    print(
        "Training:",
        X_train.shape
    )

    print(
        "Testing:",
        X_test.shape
    )

    # ========================================================
    # STANDARDISATION: TT parameters may have different numerical scales.
    # ========================================================

    scaler = StandardScaler()

    # IMPORTANT:
    # Fit ONLY on training subjects

    X_train_scaled = (
        scaler.fit_transform(
            X_train
        )
    )

    # Apply the SAME scaler to test subject

    X_test_scaled = (
        scaler.transform(
            X_test
        )
    )

    print(
        "Standardisation complete."
    )

    # ========================================================
    # PCA
    # ========================================================

    for n_components in N_COMPONENTS:

        print("\n" + "-" * 70)

        print(
            f"PCA components: "
            f"{n_components}"
        )

        # ----------------------------------------------------
        # CREATE PCA
        # ----------------------------------------------------

        pca = PCA(
            n_components=n_components,
            svd_solver="randomized",
            random_state=42
        )

        # ----------------------------------------------------
        # FIT PCA ONLY ON TRAINING DATA
        # ----------------------------------------------------

        X_train_pca = (
            pca.fit_transform(
                X_train_scaled
            )
        )

        # ----------------------------------------------------
        # TRANSFORM TEST DATA
        # ----------------------------------------------------

        X_test_pca = (
            pca.transform(
                X_test_scaled
            )
        )

        # ----------------------------------------------------
        # EXPLAINED VARIANCE
        # ----------------------------------------------------

        variance = (
            np.sum(
                pca.explained_variance_ratio_
            )
        )

        print(
            "Variance retained:",
            f"{variance:.4f}"
        )

        print(
            "Training PCA shape:",
            X_train_pca.shape
        )

        print(
            "Testing PCA shape:",
            X_test_pca.shape
        )

        # ====================================================
        # SAVE PCA DATA
        # ====================================================

        save_path = (
            RESULTS_ROOT
            / f"loso_test_{test_subject}"
            / f"pca_{n_components}"
            / "data.npz"
        )

        save_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        np.savez_compressed(

            save_path,

            # PCA features
            X_train=X_train_pca,
            X_test=X_test_pca,

            # Labels
            y_train=y_train,
            y_test=y_test,

            # Metadata
            subjects_train=subjects_train,
            subjects_test=subjects_test,

            tasks_train=tasks_train,
            tasks_test=tasks_test,

            runs_train=runs_train,
            runs_test=runs_test,

            trial_numbers_train=(
                trial_numbers_train
            ),

            trial_numbers_test=(
                trial_numbers_test
            ),

            # Information about PCA
            n_components=n_components,

            explained_variance_ratio=(
                pca.explained_variance_ratio_
            ),

            variance_retained=variance,

            # Information about TT
            tt_ranks=np.asarray(
                [15, 10]
            )
        )

        print(
            "Saved:",
            save_path
        )


print("\n")
print("=" * 70)
print("PCA PROCESSING COMPLETE")
print("=" * 70)

TT → STANDARDISATION → PCA
TT features: (5853, 5570)
Labels: (5853,)
Subjects: ['002' '005' '006' '093']
Tasks: ['auditory' 'motor' 'rest' 'somatosensory']


TEST SUBJECT: 002
Training: (4369, 5570)
Testing: (1484, 5570)
Standardisation complete.

----------------------------------------------------------------------
PCA components: 4
Variance retained: 0.2173
Training PCA shape: (4369, 4)
Testing PCA shape: (1484, 4)
Saved: ../results/vqc/pca/loso_test_002/pca_4/data.npz

----------------------------------------------------------------------
PCA components: 8
Variance retained: 0.3481
Training PCA shape: (4369, 8)
Testing PCA shape: (1484, 8)
Saved: ../results/vqc/pca/loso_test_002/pca_8/data.npz

----------------------------------------------------------------------
PCA components: 12
Variance retained: 0.4379
Training PCA shape: (4369, 12)
Testing PCA shape: (1484, 12)
Saved: ../results/vqc/pca/loso_test_002/pca_12/data.npz

----------------------------------------------------------

| Test subject |  4 PCs |  8 PCs | 12 PCs | 16 PCs |
| ------------ | -----: | -----: | -----: | -----: |
| 002          | 21.73% | 34.81% | 43.79% | 50.91% |
| 005          | 21.53% | 34.51% | 42.89% | 49.61% |
| 006          | 21.35% | 33.61% | 41.73% | 48.21% |
| 093          | 21.33% | 33.46% | 41.46% | 47.90% |


### Angle Encoding

Your PCA values are not naturally angles. They can be negative or larger than π. Therefore, before encoding, we'll map each PCA feature to a fixed angular interval:

[0,π].

We fit this scaling only on the training subject's data, then apply the same transformation to the held-out subject. This is important because the test subject must remain completely unseen during preprocessing.

In [7]:
# ============================================================
# PATHS
# ============================================================

PCA_ROOT = Path(
    "../results/vqc/pca"
)

RESULTS_ROOT = Path(
    "../results/vqc/angle_encoding"
)

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# SETTINGS
# ============================================================

SUBJECTS = [
    "002",
    "005",
    "006",
    "093"
]

N_COMPONENTS_LIST = [
    4,
    8,
    12,
    16
]

# Angle range
ANGLE_MIN = 0.0
ANGLE_MAX = np.pi

# ============================================================
# ANGLE ENCODING FUNCTION
# ============================================================

def scale_to_angles(
    X_train,
    X_test
):
    """
    Convert PCA features into rotation angles.

    Scaling is fitted ONLY on training data.

    Input:
        X_train: (n_train, n_features)
        X_test:  (n_test, n_features)

    Output:
        angles_train
        angles_test

    Values are mapped to [0, pi].
    """

    scaler = MinMaxScaler(
        feature_range=(
            ANGLE_MIN,
            ANGLE_MAX
        )
    )

    angles_train = scaler.fit_transform(
        X_train
    )

    angles_test = scaler.transform(
        X_test
    )

    return (
        angles_train,
        angles_test,
        scaler
    )

# ============================================================
# PROCESS ALL LOSO FOLDS
# ============================================================

for test_subject in SUBJECTS:

    print("\n")
    print("=" * 70)
    print(
        "TEST SUBJECT:",
        test_subject
    )
    print("=" * 70)


    for n_components in N_COMPONENTS_LIST:

        # ----------------------------------------------------
        # LOAD PCA DATA
        # ----------------------------------------------------

        path = (
            PCA_ROOT
            / f"loso_test_{test_subject}"
            / f"pca_{n_components}"
            / "data.npz"
        )


        if not path.exists():

            print(
                "WARNING: file not found:",
                path
            )

            continue


        data = np.load(
            path,
            allow_pickle=True
        )


        X_train = data["X_train"]
        X_test = data["X_test"]

        y_train = data["y_train"]
        y_test = data["y_test"]

        subjects_train = (
            data["subjects_train"]
        )

        subjects_test = (
            data["subjects_test"]
        )

        tasks_train = data["tasks_train"]
        tasks_test = data["tasks_test"]


        print("\n")
        print("-" * 70)
        print(
            f"ANGLE ENCODING: "
            f"{n_components} QUBITS"
        )
        print("-" * 70)


        print(
            "PCA training:",
            X_train.shape
        )

        print(
            "PCA testing:",
            X_test.shape
        )

        # ----------------------------------------------------
        # SCALE PCA FEATURES TO ANGLES
        # ----------------------------------------------------

        (
            angles_train,
            angles_test,
            scaler
        ) = scale_to_angles(
            X_train,
            X_test
        )

        print(
            "Angle training:",
            angles_train.shape
        )

        print(
            "Angle testing:",
            angles_test.shape
        )

        print(
            "Training angle range:",
            angles_train.min(),
            "to",
            angles_train.max()
        )

        print(
            "Testing angle range:",
            angles_test.min(),
            "to",
            angles_test.max()
        )

        # ----------------------------------------------------
        # SAVE
        # ----------------------------------------------------

        save_path = (
            RESULTS_ROOT
            / f"loso_test_{test_subject}"
            / f"angle_{n_components}"
            / "data.npz"
        )

        save_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        np.savez_compressed(

            save_path,

            angles_train=angles_train,

            angles_test=angles_test,

            y_train=y_train,

            y_test=y_test,

            subjects_train=(
                subjects_train
            ),

            subjects_test=(
                subjects_test
            ),

            tasks_train=tasks_train,

            tasks_test=tasks_test,

            angle_min=ANGLE_MIN,

            angle_max=ANGLE_MAX,

            n_qubits=n_components
        )

        print(
            "Saved:",
            save_path
        )


print("\n")
print("=" * 70)
print("ANGLE SCALING COMPLETE")
print("=" * 70)



TEST SUBJECT: 002


----------------------------------------------------------------------
ANGLE ENCODING: 4 QUBITS
----------------------------------------------------------------------
PCA training: (4369, 4)
PCA testing: (1484, 4)
Angle training: (4369, 4)
Angle testing: (1484, 4)
Training angle range: 0.0 to 3.141592653589793
Testing angle range: 0.7252783994071009 to 2.723733678200601
Saved: ../results/vqc/angle_encoding/loso_test_002/angle_4/data.npz


----------------------------------------------------------------------
ANGLE ENCODING: 8 QUBITS
----------------------------------------------------------------------
PCA training: (4369, 8)
PCA testing: (1484, 8)
Angle training: (4369, 8)
Angle testing: (1484, 8)
Training angle range: 0.0 to 3.1415926535897936
Testing angle range: 0.7252836499004556 to 2.723726133323276
Saved: ../results/vqc/angle_encoding/loso_test_002/angle_8/data.npz


----------------------------------------------------------------------
ANGLE ENCODING: 12 Q

Test:
- Test 1 — Correct number of angles
- Test 2 — Valid angle range
- Test 3 — The quantum state exists
- Test 4 — Normalisation

In [10]:
# ============================================================
# PATH
# ============================================================

ANGLE_PATH = (
    Path("../results/vqc/angle_encoding")
    / "loso_test_002"
    / "angle_8"
    / "data.npz"
)


# ============================================================
# LOAD
# ============================================================

data = np.load(
    ANGLE_PATH,
    allow_pickle=True
)


angles_train = data[
    "angles_train"
]

angles_test = data[
    "angles_test"
]


n_qubits = int(
    data["n_qubits"]
)


# ============================================================
# BASIC CHECKS
# ============================================================

print("=" * 70)
print("ANGLE ENCODING TEST")
print("=" * 70)

print(
    "Number of qubits:",
    n_qubits
)

print(
    "Training angles:",
    angles_train.shape
)

print(
    "Testing angles:",
    angles_test.shape
)


# ============================================================
# ANGLE RANGE CHECK
# ============================================================

print("\n")
print("=" * 70)
print("ANGLE RANGE")
print("=" * 70)

print(
    "Train minimum:",
    angles_train.min()
)

print(
    "Train maximum:",
    angles_train.max()
)

print(
    "Test minimum:",
    angles_test.min()
)

print(
    "Test maximum:",
    angles_test.max()
)

assert np.all(
    angles_train >= 0
)

assert np.all(
    angles_train <= np.pi + 1e-12
)

assert np.all(
    angles_test >= 0
)

assert np.all(
    angles_test <= np.pi + 1e-12
)

print(
    "PASS: all angles are in [0, pi]."
)


# ============================================================
# CHECK FEATURE → QUBIT DIMENSION
# ============================================================

assert (
    angles_train.shape[1]
    == n_qubits
)

assert (
    angles_test.shape[1]
    == n_qubits
)

print(
    "PASS: number of angles matches "
    "number of qubits."
)


# ============================================================
# CREATE QUANTUM DEVICE
# ============================================================

dev = qml.device(
    "default.qubit",
    wires=n_qubits
)


# ============================================================
# ANGLE-ENCODING CIRCUIT
# ============================================================

@qml.qnode(dev)
def angle_encoding_circuit(
    angles
):

    qml.AngleEmbedding(
        angles,
        wires=range(n_qubits),
        rotation="Y"
    )

    return qml.state()


# ============================================================
# TEST ONE SAMPLE
# ============================================================

sample = angles_train[0]


print("\n")
print("=" * 70)
print("SINGLE SAMPLE TEST")
print("=" * 70)

print(
    "Input angles:"
)

print(sample)


state = angle_encoding_circuit(
    sample
)


print(
    "Quantum state shape:",
    state.shape
)


# ============================================================
# STATE NORMALISATION
# ============================================================

norm = np.linalg.norm(
    state
)


print(
    "Quantum state norm:",
    norm
)


assert np.isclose(
    norm,
    1.0,
    atol=1e-7
)


print(
    "PASS: quantum state is normalized."
)


# ============================================================
# STATE DIMENSION
# ============================================================

expected_state_dimension = (
    2 ** n_qubits
)


print(
    "Expected state dimension:",
    expected_state_dimension
)

print(
    "Actual state dimension:",
    len(state)
)


assert (
    len(state)
    == expected_state_dimension
)


print(
    "PASS: state dimension is correct."
)


# ============================================================
# TEST MULTIPLE SAMPLES
# ============================================================

N_TEST = min(
    10,
    len(angles_train)
)


print("\n")
print("=" * 70)
print(
    f"TESTING {N_TEST} SAMPLES"
)
print("=" * 70)


for i in range(N_TEST):

    state = angle_encoding_circuit(
        angles_train[i]
    )

    norm = np.linalg.norm(
        state
    )

    assert np.isclose(
        norm,
        1.0,
        atol=1e-7
    )

    assert (
        len(state)
        == 2 ** n_qubits
    )


    print(
        f"Sample {i:2d} | "
        f"norm = {norm:.10f} | "
        f"state dimension = {len(state)} | "
        f"PASS"
    )


# ============================================================
# FINAL
# ============================================================

print("\n")
print("=" * 70)
print("ALL ANGLE ENCODING TESTS PASSED")
print("=" * 70)

ANGLE ENCODING TEST
Number of qubits: 8
Training angles: (4369, 8)
Testing angles: (1484, 8)


ANGLE RANGE
Train minimum: 0.0
Train maximum: 3.1415926535897936
Test minimum: 0.7252836499004556
Test maximum: 2.723726133323276
PASS: all angles are in [0, pi].
PASS: number of angles matches number of qubits.


SINGLE SAMPLE TEST
Input angles:
[1.02301518 2.28215129 2.48952261 2.35333407 1.92551421 2.05797408
 1.3376468  1.49961436]
Quantum state shape: (256,)
Quantum state norm: 0.9999999999999998
PASS: quantum state is normalized.
Expected state dimension: 256
Actual state dimension: 256
PASS: state dimension is correct.


TESTING 10 SAMPLES
Sample  0 | norm = 1.0000000000 | state dimension = 256 | PASS
Sample  1 | norm = 1.0000000000 | state dimension = 256 | PASS
Sample  2 | norm = 1.0000000000 | state dimension = 256 | PASS
Sample  3 | norm = 1.0000000000 | state dimension = 256 | PASS
Sample  4 | norm = 1.0000000000 | state dimension = 256 | PASS
Sample  5 | norm = 1.0000000000 | sta

### VQC

The purpose of the VQC is to determine whether the compressed representation of the MEG trial contains information that can distinguish the four tasks:
{auditory,somatosensory,motor,rest}.

8 qubits
2 trainable layers

1. Takes 32 MEG trials.
2. Runs them through the quantum circuit.
3. Gets four outputs.
4. Applies softmax.
5. Compares predictions against the true labels.
6. Calculates cross-entropy loss.
7. Uses Adam to update the VQC parameters.

Conceptually:
MEG features → VQC(θ) → prediction → loss → ∂θ/∂L → update θ.

VQC TRAINABILITY TEST

In [17]:
import pennylane as qml
from pennylane import numpy as np

N_QUBITS = 8
N_LAYERS = 2

dev = qml.device(
    "default.qubit",
    wires=N_QUBITS
)


@qml.qnode(
    dev,
    interface="autograd"
)
def circuit(x, weights):

    # -----------------------------
    # Angle encoding
    # -----------------------------

    for q in range(N_QUBITS):

        qml.RY(
            x[q],
            wires=q
        )

    # -----------------------------
    # Trainable layers
    # -----------------------------

    for layer in range(N_LAYERS):

        for q in range(N_QUBITS):

            qml.RY(
                weights[layer, q, 0],
                wires=q
            )

            qml.RZ(
                weights[layer, q, 1],
                wires=q
            )

        # Entanglement
        for q in range(N_QUBITS - 1):

            qml.CNOT(
                wires=[
                    q,
                    q + 1
                ]
            )

    return [
        qml.expval(
            qml.PauliZ(q)
        )
        for q in range(N_QUBITS)
    ]


# ============================================================
# CREATE TRAINABLE PARAMETERS
# ============================================================

weights = np.array(
    0.01 * np.random.randn(
        N_LAYERS,
        N_QUBITS,
        2
    ),
    requires_grad=True
)

x = np.array(
    np.random.uniform(
        0,
        np.pi,
        N_QUBITS
    )
)

print("=" * 70)
print("VQC TRAINABILITY TEST")
print("=" * 70)

print(
    "Weights shape:",
    weights.shape
)

print(
    "Weights requires grad:",
    weights.requires_grad
)


# ============================================================
# CIRCUIT OUTPUT
# ============================================================

output = circuit(
    x,
    weights
)

print(
    "\nCircuit output:"
)

print(output)


# ============================================================
# SIMPLE LOSS
# ============================================================

def loss_fn(weights):

    output = circuit(
        x,
        weights
    )

    return qml.math.sum(
        qml.math.stack(output)
    )


loss_before = loss_fn(weights)

print(
    "\nLoss before:",
    loss_before
)


# ============================================================
# GRADIENT
# ============================================================

gradient = qml.grad(
    loss_fn
)(weights)

print(
    "\nGradient shape:",
    gradient.shape
)

print(
    "Gradient norm:",
    np.linalg.norm(gradient)
)


# ============================================================
# TEST OPTIMIZER
# ============================================================

opt = qml.AdamOptimizer(
    stepsize=0.01
)

weights_new, loss_new = opt.step_and_cost(
    loss_fn,
    weights
)

print(
    "\nLoss after one optimizer step:",
    loss_new
)

print(
    "Weight change:",
    np.linalg.norm(
        weights_new - weights
    )
)


if np.linalg.norm(gradient) > 1e-10:

    print(
        "\nPASS: VQC has non-zero gradients."
    )

else:

    print(
        "\nFAIL: VQC gradient is zero."
    )


if np.linalg.norm(
    weights_new - weights
) > 1e-10:

    print(
        "PASS: optimizer changed "
        "the VQC parameters."
    )

else:

    print(
        "FAIL: optimizer did not "
        "change the parameters."
    )

VQC TRAINABILITY TEST
Weights shape: (2, 8, 2)
Weights requires grad: True

Circuit output:
[tensor(0.1460399, requires_grad=True), tensor(0.73781696, requires_grad=True), tensor(-0.07150067, requires_grad=True), tensor(-0.70588721, requires_grad=True), tensor(0.01444161, requires_grad=True), tensor(0.68089377, requires_grad=True), tensor(-0.01427762, requires_grad=True), tensor(0.17551632, requires_grad=True)]

Loss before: 0.9630430579654475

Gradient shape: (2, 8, 2)
Gradient norm: 1.2932659291750896

Loss after one optimizer step: 0.9630430579654475
Weight change: 0.048416096267938774

PASS: VQC has non-zero gradients.
PASS: optimizer changed the VQC parameters.


In [27]:
# ============================================================
# PATHS
# ============================================================

ANGLE_ROOT = Path(
    "../results/vqc/angle_encoding"
)

RESULTS_ROOT = Path(
    "../results/vqc/results"
)

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

EXCEL_PATH = (
    RESULTS_ROOT /
    "vqc_pca_qubit_comparison.xlsx"
)

# ============================================================
# SETTINGS
# ============================================================

SUBJECTS = [
    "002",
    "005",
    "006",
    "093"
]

QUBIT_LIST = [
    4,
    8,
    12,
    16
]

N_LAYERS = 2
N_CLASSES = 4
N_EPOCHS = 30
LEARNING_RATE = 0.05
BATCH_SIZE = 32
RANDOM_SEED = 42

TASK_NAMES = [
    "auditory",
    "somatosensory",
    "motor",
    "rest"
]

# ============================================================
# RANDOM SEED
# ============================================================

np.random.seed(
    RANDOM_SEED
)

# ============================================================
# FUNCTIONS
# ============================================================

def softmax(x):

    x = np.asarray(x)

    x = x - np.max(
        x,
        axis=-1,
        keepdims=True
    )

    exp_x = np.exp(x)

    return (
        exp_x
        /
        np.sum(
            exp_x,
            axis=-1,
            keepdims=True
        )
    )

def initialise_weights(n_qubits):

    return (
        0.01
        * np.random.randn(
            N_LAYERS,
            n_qubits,
            3
        )
    )

def create_quantum_circuit(n_qubits):

    dev = qml.device(
        "default.qubit",
        wires=n_qubits
    )

    @qml.qnode(dev)
    def quantum_circuit(
        x,
        weights
    ):

        # --------------------------------------------------------
        # ANGLE ENCODING
        # --------------------------------------------------------

        qml.AngleEmbedding(
            x,
            wires=range(n_qubits),
            rotation="Y"
        )

        # --------------------------------------------------------
        # TRAINABLE VQC LAYERS
        # --------------------------------------------------------

        for layer in range(N_LAYERS):

            # Trainable rotations
            for qubit in range(n_qubits):

                # Each Rot gate has three trainable parameters.
                # So 2 layers × 8 qubits × 3 = 48 trainable parameters.
                qml.Rot(
                    weights[layer, qubit, 0],
                    weights[layer, qubit, 1],
                    weights[layer, qubit, 2],
                    wires=qubit
                )

            # ----------------------------------------------------
            # ENTANGLEMENT
            # ----------------------------------------------------

            for qubit in range(
                n_qubits - 1
            ):

                qml.CNOT(
                    wires=[
                        qubit,
                        qubit + 1
                    ]
                )


        # --------------------------------------------------------
        # MEASUREMENTS
        # --------------------------------------------------------

        return [
            qml.expval(
                qml.PauliZ(i)
            )
            for i in range(N_CLASSES)
        ]

    return quantum_circuit

# ============================================================
# FORWARD PASS
# ============================================================

def predict_logits(
    X,
    weights,
    quantum_circuit
):

    outputs = []

    for sample in X:

        result = quantum_circuit(
            sample,
            weights
        )

        outputs.append(
            np.asarray(result)
        )

    return np.asarray(
        outputs
    )


# ============================================================
# CROSS-ENTROPY
# ============================================================

def cross_entropy(
    probabilities,
    labels
):

    probabilities = np.clip(
        probabilities,
        1e-10,
        1.0
    )

    losses = -np.log(
        probabilities[
            np.arange(
                len(labels)
            ),
            labels
        ]
    )

    return np.mean(
        losses
    )

# ============================================================
# TRAIN VQC
# ============================================================

def train_vqc(
    X_train,
    y_train,
    n_qubits
):

    quantum_circuit = (
        create_quantum_circuit(
            n_qubits
        )
    )

    weights = initialise_weights(n_qubits)

    # OPTIMISER
    opt = qml.AdamOptimizer(
        stepsize=LEARNING_RATE
    )

    # COST FUNCTION
    def cost_fn(
        weights,
        X_batch,
        y_batch
    ):

        logits = predict_logits(
            X_batch,
            weights,
            quantum_circuit
        )

        probabilities = softmax(
            logits
        )

        return cross_entropy(
            probabilities,
            y_batch
        )


    # --------------------------------------------------------
    # TRAINING LOOP
    # --------------------------------------------------------

    loss_history = []

    for epoch in range(
        N_EPOCHS
    ):

        # Random mini-batch
        batch_size = min(
            BATCH_SIZE,
            len(X_train)
        )

        batch_indices = np.random.choice(
            len(X_train),
            size=batch_size,
            replace=False
        )

        X_batch = X_train[
            batch_indices
        ]

        y_batch = y_train[
            batch_indices
        ]

        # Update weights
        weights, loss = opt.step_and_cost(
            lambda w:
                cost_fn(
                    w,
                    X_batch,
                    y_batch
                ),
            weights
        )

        loss_history.append(
            float(loss)
        )

        if (
            epoch == 0
            or
            (epoch + 1) % 5 == 0
        ):

            print(
                f"Epoch "
                f"{epoch + 1:3d}/{N_EPOCHS} "
                f"| Loss = "
                f"{loss:.6f}"
            )

    return (
        weights,
        quantum_circuit,
        loss_history
    )

# ============================================================
# PREDICTION
# ============================================================

def predict(
    X,
    weights,
    quantum_circuit
):

    logits = predict_logits(
        X,
        weights,
        quantum_circuit
    )

    probabilities = softmax(
        logits
    )

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    return (
        predictions,
        probabilities
    )

all_results = []

for n_qubits in QUBIT_LIST:

    print("\n")
    print("=" * 80)
    print(
        f"VQC EXPERIMENT: "
        f"{n_qubits} QUBITS"
    )
    print("=" * 80)

    # Check angle encoding directory

    for test_subject in SUBJECTS:

        print("\n")
        print("-" * 80)

        print(
            f"QUBITS = {n_qubits}"
        )

        print(
            f"TEST SUBJECT = "
            f"{test_subject}"
        )

        print("-" * 80)

        # Load corresponding angle data

        angle_path = (
            ANGLE_ROOT
            /
            f"loso_test_{test_subject}"
            /
            f"angle_{n_qubits}"
            /
            "data.npz"
        )

        if not angle_path.exists():

            raise FileNotFoundError(
                f"\nMissing angle file:\n"
                f"{angle_path}\n\n"
                f"Make sure PCA/angle encoding "
                f"has been generated for "
                f"{n_qubits} components."
            )

        data = np.load(
            angle_path,
            allow_pickle=True
        )

        X_train = data[
            "angles_train"
        ]

        X_test = data[
            "angles_test"
        ]

        y_train = data[
            "y_train"
        ]

        y_test = data[
            "y_test"
        ]

        # Check dimensions

        if X_train.shape[1] != n_qubits:

            raise ValueError(
                f"Expected {n_qubits} "
                f"features but received "
                f"{X_train.shape[1]}"
            )

        print(
            "Training:",
            X_train.shape
        )

        print(
            "Testing:",
            X_test.shape
        )

        print(
            "Training labels:",
            np.unique(y_train)
        )

        print(
            "Testing labels:",
            np.unique(y_test)
        )

        # TRAIN

        print("\n")
        print(
            "Training VQC..."
        )

        (
            weights,
            quantum_circuit,
            loss_history
        ) = train_vqc(
            X_train,
            y_train,
            n_qubits
        )

        # TEST

        print("\n")
        print(
            "Testing VQC..."
        )

        predictions, probabilities = (
            predict(
                X_test,
                weights,
                quantum_circuit
            )
        )

        # METRICS

        accuracy = accuracy_score(
            y_test,
            predictions
        )

        balanced_accuracy = (
            balanced_accuracy_score(
                y_test,
                predictions
            )
        )

        macro_f1 = f1_score(
            y_test,
            predictions,
            average="macro",
            zero_division=0
        )

        weighted_f1 = f1_score(
            y_test,
            predictions,
            average="weighted",
            zero_division=0
        )

        cm = confusion_matrix(
            y_test,
            predictions,
            labels=[
                0,
                1,
                2,
                3
            ]
        )

        print(
            "\nTest accuracy:",
            f"{accuracy:.4f}"
        )

        print(
            "Balanced accuracy:",
            f"{balanced_accuracy:.4f}"
        )

        print(
            "Macro F1:",
            f"{macro_f1:.4f}"
        )

        print(
            "Weighted F1:",
            f"{weighted_f1:.4f}"
        )

        print(
            "\nConfusion matrix:"
        )

        print(cm)

        # SAVE INDIVIDUAL RESULT

        result_dir = (
            RESULTS_ROOT
            /
            f"qubits_{n_qubits}"
        )

        result_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        save_path = (
            result_dir
            /
            f"loso_test_{test_subject}.npz"
        )

        np.savez_compressed(

            save_path,
            test_subject=test_subject,
            predictions=predictions,
            probabilities=probabilities,
            y_test=y_test,
            accuracy=accuracy,
            balanced_accuracy=balanced_accuracy,
            macro_f1=macro_f1,
            weighted_f1=weighted_f1,
            confusion_matrix=cm,
            weights=weights,
            loss_history=np.asarray(
                loss_history
            ),
            n_qubits=n_qubits,
            n_layers=N_LAYERS
        )

        print(
            "Saved:",
            save_path
        )

        # STORE ROW FOR EXCEL

        all_results.append({

            "subject": test_subject,
            "n_qubits": n_qubits,
            "accuracy": accuracy,
            "balanced_accuracy":
                balanced_accuracy,
            "macro_f1":
                macro_f1,
            "weighted_f1":
                weighted_f1,
            "final_training_loss":
                loss_history[-1]
        })

# CONVERT RESULTS TO DATAFRAME

results_df = pd.DataFrame(
    all_results
)
    
# LOSO SUMMARY

summary_df = (
    results_df
    .groupby(
        "n_qubits"
    )
    .agg({

        "accuracy":
            ["mean", "std"],

        "balanced_accuracy":
            ["mean", "std"],

        "macro_f1":
            ["mean", "std"],

        "weighted_f1":
            ["mean", "std"],

        "final_training_loss":
            ["mean", "std"]

    })
    .reset_index()
)

# Flatten column names

summary_df.columns = [

    "n_qubits",

    "accuracy_mean",
    "accuracy_std",

    "balanced_accuracy_mean",
    "balanced_accuracy_std",

    "macro_f1_mean",
    "macro_f1_std",

    "weighted_f1_mean",
    "weighted_f1_std",

    "loss_mean",
    "loss_std"

]

# SAVE EXCEL

with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl"
) as writer:

    results_df.to_excel(
        writer,
        sheet_name="LOSO Results",
        index=False
    )

    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )


print("\n")
print("=" * 80)
print("EXCEL RESULTS SAVED")
print("=" * 80)

print(
    EXCEL_PATH
)

# PRINT SUMMARY

print("\n")
print("=" * 80)
print("VQC QUANTUM DIMENSION SUMMARY")
print("=" * 80)

print(
    summary_df.to_string(
        index=False
    )
)

# ============================================================
# PLOT 1:
# MEAN ACCURACY VS NUMBER OF QUBITS
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    summary_df["n_qubits"],
    summary_df["accuracy_mean"],
    yerr=summary_df["accuracy_std"],
    marker="o",
    capsize=5
)

plt.axhline(
    0.25,
    linestyle="--",
    label="Random baseline (25%)"
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "LOSO accuracy"
)

plt.title(
    "VQC Accuracy vs Number of Qubits"
)

plt.xticks(
    QUBIT_LIST
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()


accuracy_plot = (
    RESULTS_ROOT /
    "accuracy_vs_qubits.png"
)

plt.savefig(
    accuracy_plot,
    dpi=300
)

plt.close()


# ============================================================
# PLOT 2:
# MACRO F1 VS NUMBER OF QUBITS
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    summary_df["n_qubits"],
    summary_df["macro_f1_mean"],
    yerr=summary_df["macro_f1_std"],
    marker="o",
    capsize=5
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "Macro F1"
)

plt.title(
    "VQC Macro F1 vs Number of Qubits"
)

plt.xticks(
    QUBIT_LIST
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()


f1_plot = (
    RESULTS_ROOT /
    "macro_f1_vs_qubits.png"
)

plt.savefig(
    f1_plot,
    dpi=300
)

plt.close()


# ============================================================
# PLOT 3:
# BALANCED ACCURACY VS NUMBER OF QUBITS
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    summary_df["n_qubits"],
    summary_df["balanced_accuracy_mean"],
    yerr=summary_df["balanced_accuracy_std"],
    marker="o",
    capsize=5
)

plt.axhline(
    0.25,
    linestyle="--",
    label="Random baseline (25%)"
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "Balanced accuracy"
)

plt.title(
    "VQC Balanced Accuracy vs Number of Qubits"
)

plt.xticks(
    QUBIT_LIST
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()


balanced_plot = (
    RESULTS_ROOT /
    "balanced_accuracy_vs_qubits.png"
)

plt.savefig(
    balanced_plot,
    dpi=300
)

plt.close()


# ============================================================
# PLOT 4:
# ACCURACY FOR EACH SUBJECT
# ============================================================

plt.figure(
    figsize=(9, 6)
)

for subject in SUBJECTS:

    subject_data = results_df[
        results_df["subject"] == subject
    ]

    plt.plot(
        subject_data["n_qubits"],
        subject_data["accuracy"],
        marker="o",
        label=f"Subject {subject}"
    )


plt.axhline(
    0.25,
    linestyle="--",
    label="Random baseline (25%)"
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "LOSO accuracy"
)

plt.title(
    "LOSO Accuracy Across Quantum Dimensions"
)

plt.xticks(
    QUBIT_LIST
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()


subject_plot = (
    RESULTS_ROOT /
    "accuracy_by_subject.png"
)

plt.savefig(
    subject_plot,
    dpi=300
)

plt.close()


# ============================================================
# COMPLETE
# ============================================================

print("\n")
print("=" * 80)
print("ALL VQC EXPERIMENTS COMPLETE")
print("=" * 80)

print(
    "Excel:",
    EXCEL_PATH
)

print(
    "Plots saved in:",
    RESULTS_ROOT
)



VQC EXPERIMENT: 4 QUBITS


--------------------------------------------------------------------------------
QUBITS = 4
TEST SUBJECT = 002
--------------------------------------------------------------------------------
Training: (4369, 4)
Testing: (1484, 4)
Training labels: [0 1 2 3]
Testing labels: [0 1 2 3]


Training VQC...
Epoch   1/30 | Loss = 1.394091
Epoch   5/30 | Loss = 1.284441


/home/master/.local/lib/python3.13/site-packages/pennylane/_grad/grad.py:377: UserWarning: Attempted to differentiate a function with no trainable parameters. If this is unintended, please add trainable parameters via the 'requires_grad' attribute or 'argnums' keyword.
  warnings.warn(


Epoch  10/30 | Loss = 1.563577
Epoch  15/30 | Loss = 1.605180
Epoch  20/30 | Loss = 1.500097
Epoch  25/30 | Loss = 1.534501
Epoch  30/30 | Loss = 1.450115


Testing VQC...

Test accuracy: 0.2749
Balanced accuracy: 0.2439
Macro F1: 0.1683
Weighted F1: 0.1932

Confusion matrix:
[[276   0   0 124]
 [259   0   0 146]
 [136   0   0  81]
 [330   0   0 132]]
Saved: ../results/vqc/results/qubits_4/loso_test_002.npz


--------------------------------------------------------------------------------
QUBITS = 4
TEST SUBJECT = 005
--------------------------------------------------------------------------------
Training: (4369, 4)
Testing: (1484, 4)
Training labels: [0 1 2 3]
Testing labels: [0 1 2 3]


Training VQC...
Epoch   1/30 | Loss = 1.494132


/home/master/.local/lib/python3.13/site-packages/pennylane/_grad/grad.py:377: UserWarning: Attempted to differentiate a function with no trainable parameters. If this is unintended, please add trainable parameters via the 'requires_grad' attribute or 'argnums' keyword.
  warnings.warn(


Epoch   5/30 | Loss = 1.411627
Epoch  10/30 | Loss = 1.377108
Epoch  15/30 | Loss = 1.455983
Epoch  20/30 | Loss = 1.489805
Epoch  25/30 | Loss = 1.642017
Epoch  30/30 | Loss = 1.506575


Testing VQC...

Test accuracy: 0.2426
Balanced accuracy: 0.2227
Macro F1: 0.1291
Weighted F1: 0.1423

Confusion matrix:
[[317   0   1  82]
 [288   1   0 111]
 [216   0   0  30]
 [396   0   0  42]]
Saved: ../results/vqc/results/qubits_4/loso_test_005.npz


--------------------------------------------------------------------------------
QUBITS = 4
TEST SUBJECT = 006
--------------------------------------------------------------------------------
Training: (4442, 4)
Testing: (1411, 4)
Training labels: [0 1 2 3]
Testing labels: [0 1 2 3]


Training VQC...
Epoch   1/30 | Loss = 1.361486


...

Epoch   1/30 | Loss = 1.600104
Epoch   5/30 | Loss = 1.425478
Epoch  10/30 | Loss = 1.597290
Epoch  15/30 | Loss = 1.549083
Epoch  20/30 | Loss = 1.399442
Epoch  25/30 | Loss = 1.538251
Epoch  30/30 | Loss = 1.621661


Testing VQC...

Test accuracy: 0.2412
Balanced accuracy: 0.2221
Macro F1: 0.1224
Weighted F1: 0.1343

Confusion matrix:
[[327   0   1  72]
 [306   1   0  93]
 [222   0   0  24]
 [408   0   0  30]]
Saved: ../results/vqc/results/qubits_16/loso_test_005.npz


--------------------------------------------------------------------------------
QUBITS = 16
TEST SUBJECT = 006
--------------------------------------------------------------------------------
Training: (4442, 16)
Testing: (1411, 16)
Training labels: [0 1 2 3]
Testing labels: [0 1 2 3]


Training VQC...


/home/master/.local/lib/python3.13/site-packages/pennylane/_grad/grad.py:377: UserWarning: Attempted to differentiate a function with no trainable parameters. If this is unintended, please add trainable parameters via the 'requires_grad' attribute or 'argnums' keyword.
  warnings.warn(


Epoch   1/30 | Loss = 1.415450
Epoch   5/30 | Loss = 1.384627
Epoch  10/30 | Loss = 1.333443
Epoch  15/30 | Loss = 1.365740
Epoch  20/30 | Loss = 1.434240
Epoch  25/30 | Loss = 1.402113
Epoch  30/30 | Loss = 1.397096


Testing VQC...

Test accuracy: 0.2481
Balanced accuracy: 0.2132
Macro F1: 0.1999
Weighted F1: 0.2310

Confusion matrix:
[[ 64 212  12 112]
 [ 86 164  21 136]
 [ 42  68   3  53]
 [ 80 230   9 119]]
Saved: ../results/vqc/results/qubits_16/loso_test_006.npz


--------------------------------------------------------------------------------
QUBITS = 16
TEST SUBJECT = 093
--------------------------------------------------------------------------------
Training: (4379, 16)
Testing: (1474, 16)
Training labels: [0 1 2 3]
Testing labels: [0 1 2 3]


Training VQC...


/home/master/.local/lib/python3.13/site-packages/pennylane/_grad/grad.py:377: UserWarning: Attempted to differentiate a function with no trainable parameters. If this is unintended, please add trainable parameters via the 'requires_grad' attribute or 'argnums' keyword.
  warnings.warn(


Epoch   1/30 | Loss = 1.442711
Epoch   5/30 | Loss = 1.501205
Epoch  10/30 | Loss = 1.586023
Epoch  15/30 | Loss = 1.535556
Epoch  20/30 | Loss = 1.536313
Epoch  25/30 | Loss = 1.444720
Epoch  30/30 | Loss = 1.543363


Testing VQC...

Test accuracy: 0.2727
Balanced accuracy: 0.2510
Macro F1: 0.1140
Weighted F1: 0.1242

Confusion matrix:
[[397   0   0   3]
 [377   0   0  40]
 [219   0   0   2]
 [431   0   0   5]]
Saved: ../results/vqc/results/qubits_16/loso_test_093.npz


EXCEL RESULTS SAVED
../results/vqc/results/vqc_pca_qubit_comparison.xlsx


VQC QUANTUM DIMENSION SUMMARY
 n_qubits  accuracy_mean  accuracy_std  balanced_accuracy_mean  balanced_accuracy_std  macro_f1_mean  macro_f1_std  weighted_f1_mean  weighted_f1_std  loss_mean  loss_std
        4       0.259921      0.016655                0.233124               0.017513       0.152998      0.038141          0.172713         0.047584   1.417151  0.077423
        8       0.260495      0.017005                0.233656               

In [8]:
# ============================================================
# PATHS
# ============================================================

PCA_ROOT = Path("../data/vqc/pca")
RESULTS_ROOT = Path("../results/vqc/diagnostics")

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# SETTINGS
# ============================================================

N_QUBITS_LIST = [4, 8, 12, 16]

N_LAYERS = 2

N_EPOCHS = 30

LEARNING_RATE = 0.01

SEED = 42

SUBJECTS = [
    "002",
    "005",
    "006",
    "093"
]

TASK_NAMES = [
    "auditory",
    "somatosensory",
    "motor",
    "rest"
]

N_CLASSES = 4


# ============================================================
# REPRODUCIBILITY
# ============================================================

np.random.seed(SEED)


# ============================================================
# VQC
# ============================================================

def make_vqc(
    n_qubits,
    n_layers
):

    dev = qml.device(
        "default.qubit",
        wires=n_qubits
    )


    @qml.qnode(
        dev,
        interface="autograd"
    )
    def circuit(
        angles,
        weights
    ):

        # ----------------------------------------------------
        # Angle encoding
        # ----------------------------------------------------

        for q in range(n_qubits):

            qml.RY(
                angles[q],
                wires=q
            )


        # ----------------------------------------------------
        # Trainable variational layers
        # ----------------------------------------------------

        for layer in range(n_layers):

            for q in range(n_qubits):

                qml.RY(
                    weights[layer, q, 0],
                    wires=q
                )

                qml.RZ(
                    weights[layer, q, 1],
                    wires=q
                )


            # Ring entanglement

            for q in range(n_qubits - 1):

                qml.CNOT(
                    wires=[
                        q,
                        q + 1
                    ]
                )

            qml.CNOT(
                wires=[
                    n_qubits - 1,
                    0
                ]
            )


        # ----------------------------------------------------
        # Measurements
        # ----------------------------------------------------

        return [
            qml.expval(
                qml.PauliZ(q)
            )
            for q in range(n_qubits)
        ]

    return circuit


# ============================================================
# CLASSIFIER
# ============================================================

def logits_from_output(
    output
):

    """
    Convert expectation values into
    four class logits.

    For 4 classes we use groups of
    qubit expectation values.

    """

    output = np.asarray(
        output,
        dtype=float
    )

    # --------------------------------------------------------
    # Split available qubits into four groups
    # --------------------------------------------------------

    groups = np.array_split(
        output,
        N_CLASSES
    )

    logits = np.array([
        np.mean(group)
        for group in groups
    ])

    return logits


# ============================================================
# LOSS
# ============================================================

def cross_entropy_loss(
    logits,
    label
):

    logits = np.asarray(
        logits
    )

    # Stable softmax

    shifted = (
        logits -
        np.max(logits)
    )

    exp_logits = np.exp(
        shifted
    )

    probabilities = (
        exp_logits /
        np.sum(exp_logits)
    )

    return -np.log(
        probabilities[label] +
        1e-12
    )


# ============================================================
# FORWARD PASS
# ============================================================

def predict_dataset(
    circuit,
    X,
    weights
):

    predictions = []

    losses = []

    for x, label in X:

        output = circuit(
            x,
            weights
        )

        output = np.asarray(
            output,
            dtype=float
        )

        logits = logits_from_output(
            output
        )

        prediction = np.argmax(
            logits
        )

        predictions.append(
            prediction
        )

        losses.append(
            cross_entropy_loss(
                logits,
                label
            )
        )

    return (
        np.asarray(predictions),
        np.asarray(losses)
    )


# ============================================================
# TRAINING FUNCTION
# ============================================================

def train_vqc(
    circuit,
    X_train,
    y_train,
    weights
):

    optimizer = qml.AdamOptimizer(
        stepsize=LEARNING_RATE
    )


    # --------------------------------------------------------
    # Dataset loss
    # --------------------------------------------------------

    def loss_function(
        w
    ):

        losses = []

        for i in range(
            len(X_train)
        ):

            output = circuit(
                X_train[i],
                w
            )

            output = qml.math.stack(
                output
            )

            # Convert qubit outputs into
            # four class logits

            groups = qml.math.reshape(
                output,
                (
                    N_CLASSES,
                    -1
                )
            )

            logits = qml.math.mean(
                groups,
                axis=1
            )

            log_probs = (
                logits -
                qml.math.log(
                    qml.math.sum(
                        qml.math.exp(logits)
                    )
                )
            )

            losses.append(
                -log_probs[
                    y_train[i]
                ]
            )

        return qml.math.mean(
            qml.math.stack(
                losses
            )
        )


    # --------------------------------------------------------
    # Initial loss
    # --------------------------------------------------------

    initial_loss = float(
        loss_function(
            weights
        )
    )


    # --------------------------------------------------------
    # Gradient before training
    # --------------------------------------------------------

    grad_fn = qml.grad(
        loss_function
    )

    initial_gradient = grad_fn(
        weights
    )

    initial_gradient_norm = float(
        np.linalg.norm(
            np.asarray(
                initial_gradient
            )
        )
    )


    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    loss_history = []

    for epoch in range(
        N_EPOCHS
    ):

        weights, loss = optimizer.step_and_cost(
            loss_function,
            weights
        )

        loss_value = float(
            loss
        )

        loss_history.append(
            loss_value
        )

        if (
            epoch == 0
            or
            (epoch + 1) % 5 == 0
        ):

            print(
                f"Epoch "
                f"{epoch + 1:2d}/"
                f"{N_EPOCHS} | "
                f"Loss = "
                f"{loss_value:.6f}"
            )


    # --------------------------------------------------------
    # Final loss
    # --------------------------------------------------------

    final_loss = float(
        loss_function(
            weights
        )
    )


    return (
        weights,
        initial_loss,
        final_loss,
        initial_gradient_norm,
        np.asarray(loss_history)
    )


# ============================================================
# MAIN EXPERIMENT
# ============================================================

summary_rows = []


for n_qubits in N_QUBITS_LIST:

    print("\n")
    print("=" * 80)
    print(
        f"VQC DIAGNOSTIC: "
        f"{n_qubits} QUBITS"
    )
    print("=" * 80)


    # --------------------------------------------------------
    # Create VQC
    # --------------------------------------------------------

    circuit = make_vqc(
        n_qubits,
        N_LAYERS
    )


    for test_subject in SUBJECTS:

        print("\n")
        print("-" * 80)

        print(
            f"TEST SUBJECT: "
            f"{test_subject}"
        )

        print("-" * 80)


        # ====================================================
        # LOAD PCA DATA
        # ====================================================

        pca_path = (
            PCA_ROOT
            / f"loso_test_{test_subject}"
            / f"pca_{n_qubits}"
            / "data.npz"
        )


        if not pca_path.exists():

            print(
                "WARNING: PCA file not found:"
            )

            print(
                pca_path
            )

            continue


        data = np.load(
            pca_path,
            allow_pickle=True
        )

        X_train = data[
            "X_train_pca"
        ]

        X_test = data[
            "X_test_pca"
        ]

        y_train = data[
            "y_train"
        ]

        y_test = data[
            "y_test"
        ]


        print(
            "Training:",
            X_train.shape
        )

        print(
            "Testing:",
            X_test.shape
        )


        # ====================================================
        # ANGLE ENCODING
        # ====================================================

        # PCA values need to be mapped
        # into [0, pi].

        train_min = X_train.min(
            axis=0
        )

        train_max = X_train.max(
            axis=0
        )


        denominator = (
            train_max -
            train_min
        )

        denominator[
            denominator == 0
        ] = 1.0


        angles_train = (
            X_train -
            train_min
        ) / denominator


        angles_train = (
            angles_train *
            np.pi
        )


        angles_test = (
            X_test -
            train_min
        ) / denominator


        angles_test = (
            angles_test *
            np.pi
        )


        # Prevent numerical overflow

        angles_train = np.clip(
            angles_train,
            0,
            np.pi
        )

        angles_test = np.clip(
            angles_test,
            0,
            np.pi
        )


        # ====================================================
        # CREATE INITIAL WEIGHTS
        # ====================================================

        rng = np.random.default_rng(
            SEED
        )

        initial_weights = qml.numpy.array(
            rng.normal(
                0,
                0.05,
                size=(
                    N_LAYERS,
                    n_qubits,
                    2
                )
            ),
            requires_grad=True
        )


        print(
            "Initial weights shape:",
            initial_weights.shape
        )


        # ====================================================
        # TRAINING DATA FORMAT
        # ====================================================

        train_data = list(
            zip(
                angles_train,
                y_train
            )
        )


        test_data = list(
            zip(
                angles_test,
                y_test
            )
        )


        # ====================================================
        # TRAIN VQC
        # ====================================================

        print(
            "\nTraining VQC..."
        )


        (
            final_weights,
            initial_loss,
            final_loss,
            gradient_norm,
            loss_history
        ) = train_vqc(
            circuit,
            train_data,
            y_train,
            initial_weights
        )


        # ====================================================
        # WEIGHT CHANGE
        # ====================================================

        weight_change = float(
            np.linalg.norm(
                np.asarray(
                    final_weights
                ) -
                np.asarray(
                    initial_weights
                )
            )
        )


        # ====================================================
        # TRAINING PREDICTIONS
        # ====================================================

        train_predictions = []
        train_losses = []


        for i in range(
            len(angles_train)
        ):

            output = circuit(
                angles_train[i],
                final_weights
            )

            output = np.asarray(
                output,
                dtype=float
            )

            logits = logits_from_output(
                output
            )

            prediction = np.argmax(
                logits
            )

            train_predictions.append(
                prediction
            )

            train_losses.append(
                cross_entropy_loss(
                    logits,
                    y_train[i]
                )
            )


        train_predictions = np.asarray(
            train_predictions
        )


        # ====================================================
        # TEST PREDICTIONS
        # ====================================================

        test_predictions = []
        test_losses = []


        for i in range(
            len(angles_test)
        ):

            output = circuit(
                angles_test[i],
                final_weights
            )

            output = np.asarray(
                output,
                dtype=float
            )

            logits = logits_from_output(
                output
            )

            prediction = np.argmax(
                logits
            )

            test_predictions.append(
                prediction
            )

            test_losses.append(
                cross_entropy_loss(
                    logits,
                    y_test[i]
                )
            )


        test_predictions = np.asarray(
            test_predictions
        )


        # ====================================================
        # METRICS
        # ====================================================

        train_accuracy = (
            accuracy_score(
                y_train,
                train_predictions
            )
        )


        test_accuracy = (
            accuracy_score(
                y_test,
                test_predictions
            )
        )


        balanced_accuracy = (
            balanced_accuracy_score(
                y_test,
                test_predictions
            )
        )


        macro_f1 = (
            f1_score(
                y_test,
                test_predictions,
                average="macro",
                zero_division=0
            )
        )


        cm = confusion_matrix(
            y_test,
            test_predictions,
            labels=np.arange(
                N_CLASSES
            )
        )


        # ====================================================
        # PREDICTED CLASS DISTRIBUTION
        # ====================================================

        predicted_counts = np.bincount(
            test_predictions,
            minlength=N_CLASSES
        )


        # ====================================================
        # PRINT RESULTS
        # ====================================================

        print("\n")
        print(
            "=" * 70
        )

        print(
            "DIAGNOSTIC RESULTS"
        )

        print(
            "=" * 70
        )


        print(
            "Initial loss:",
            initial_loss
        )

        print(
            "Final loss:",
            final_loss
        )

        print(
            "Gradient norm:",
            gradient_norm
        )

        print(
            "Weight-change norm:",
            weight_change
        )

        print(
            "Training accuracy:",
            train_accuracy
        )

        print(
            "Test accuracy:",
            test_accuracy
        )

        print(
            "Balanced accuracy:",
            balanced_accuracy
        )

        print(
            "Macro F1:",
            macro_f1
        )


        print(
            "\nPredicted class distribution:"
        )

        for class_id, count in enumerate(
            predicted_counts
        ):

            print(
                f"  {TASK_NAMES[class_id]:15s}: "
                f"{count}"
            )


        print(
            "\nConfusion matrix:"
        )

        print(
            cm
        )


        # ====================================================
        # SAVE DETAILED RESULT
        # ====================================================

        save_dir = (
            RESULTS_ROOT
            / f"qubits_{n_qubits}"
        )

        save_dir.mkdir(
            parents=True,
            exist_ok=True
        )


        np.savez_compressed(

            save_dir
            / f"loso_test_{test_subject}.npz",

            initial_weights=np.asarray(
                initial_weights
            ),

            final_weights=np.asarray(
                final_weights
            ),

            loss_history=loss_history,

            initial_loss=initial_loss,

            final_loss=final_loss,

            gradient_norm=gradient_norm,

            weight_change=weight_change,

            train_accuracy=train_accuracy,

            test_accuracy=test_accuracy,

            balanced_accuracy=balanced_accuracy,

            macro_f1=macro_f1,

            confusion_matrix=cm,

            predicted_class_distribution=predicted_counts,

            train_predictions=train_predictions,

            test_predictions=test_predictions
        )


        # ====================================================
        # SUMMARY ROW
        # ====================================================

        summary_rows.append({

            "test_subject":
                test_subject,

            "n_qubits":
                n_qubits,

            "initial_loss":
                initial_loss,

            "final_loss":
                final_loss,

            "gradient_norm":
                gradient_norm,

            "weight_change":
                weight_change,

            "train_accuracy":
                train_accuracy,

            "test_accuracy":
                test_accuracy,

            "balanced_accuracy":
                balanced_accuracy,

            "macro_f1":
                macro_f1,

            "pred_auditory":
                predicted_counts[0],

            "pred_somatosensory":
                predicted_counts[1],

            "pred_motor":
                predicted_counts[2],

            "pred_rest":
                predicted_counts[3],

            "cm_auditory_auditory":
                cm[0, 0],

            "cm_auditory_somatosensory":
                cm[0, 1],

            "cm_auditory_motor":
                cm[0, 2],

            "cm_auditory_rest":
                cm[0, 3],

            "cm_somatosensory_auditory":
                cm[1, 0],

            "cm_somatosensory_somatosensory":
                cm[1, 1],

            "cm_somatosensory_motor":
                cm[1, 2],

            "cm_somatosensory_rest":
                cm[1, 3],

            "cm_motor_auditory":
                cm[2, 0],

            "cm_motor_somatosensory":
                cm[2, 1],

            "cm_motor_motor":
                cm[2, 2],

            "cm_motor_rest":
                cm[2, 3],

            "cm_rest_auditory":
                cm[3, 0],

            "cm_rest_somatosensory":
                cm[3, 1],

            "cm_rest_motor":
                cm[3, 2],

            "cm_rest_rest":
                cm[3, 3]
        })


# ============================================================
# SUMMARY DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    summary_rows
)


# ============================================================
# SAVE EXCEL
# ============================================================

excel_path = (
    RESULTS_ROOT /
    "vqc_diagnostic_results.xlsx"
)


with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    results_df.to_excel(
        writer,
        sheet_name="All Results",
        index=False
    )


    # --------------------------------------------------------
    # Average results by qubit count
    # --------------------------------------------------------

    qubit_summary = (
        results_df
        .groupby("n_qubits")
        [
            [
                "initial_loss",
                "final_loss",
                "gradient_norm",
                "weight_change",
                "train_accuracy",
                "test_accuracy",
                "balanced_accuracy",
                "macro_f1"
            ]
        ]
        .agg(
            ["mean", "std"]
        )
    )


    qubit_summary.to_excel(
        writer,
        sheet_name="Qubit Summary"
    )


print("\n")
print("=" * 80)
print("VQC DIAGNOSTIC COMPLETE")
print("=" * 80)

print(
    "Saved Excel:"
)

print(
    excel_path
)



VQC DIAGNOSTIC: 4 QUBITS


--------------------------------------------------------------------------------
TEST SUBJECT: 002
--------------------------------------------------------------------------------


KeyError: 'X_train_pca is not a file in the archive'